# 🔬 Task 3.2: BERTopic Hyperparameter Tuning

**Mục tiêu:** Tìm hyperparameters tối ưu cho BERTopic trên Vietnamese Tech Dataset

**Author:** Member 3 (ML Engineer)  
**Date:** 2026-04-04

---

## 📋 Workflow Overview

```
1. Setup & Install          → Packages + GPU check
2. Load Data                → Preprocessed documents
3. Compute Embeddings       → PhoBERT (CACHE để reuse)
4. Define Experiment Grid   → Hyperparameter combinations
5. Run Experiments          → Train + evaluate 10 configs
6. Analyze Results          → Select best config
7. Train Final Model        → Full dataset với best params
8. Save Results             → Model + metrics
```

**⏱️ Estimated Time: 60-90 minutes**

---
## 📦 Cell 1: Setup & Install Packages

**Mục đích:** Cài packages và kiểm tra GPU

In [ ]:
# ===============================================
# CELL 1: SETUP & INSTALL PACKAGES
# ===============================================

print("📦 Installing required packages...")
print("="*50)

# Install packages
!pip install -q bertopic==0.16.0
!pip install -q transformers==4.36.0
!pip install -q torch==2.1.0
!pip install -q umap-learn==0.5.5
!pip install -q hdbscan==0.8.33
!pip install -q gensim==4.3.2
!pip install -q sentence-transformers==2.2.2
!pip install -q tqdm pandas numpy scikit-learn

print("\n✅ Installation complete!")

# Check GPU
import torch
print("\n" + "="*50)
print("🔧 SYSTEM INFO")
print("="*50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Enable in Settings → Accelerator → GPU T4 x2")

---
## 📊 Cell 2: Load Data

**Mục đích:** Load preprocessed documents từ Task 2

In [ ]:
# ===============================================
# CELL 2: LOAD DATA
# ===============================================

import pandas as pd
import os

print("📊 LOADING DATA")
print("="*50)

# ============ CẦN SỬA PATH NÀY ============
# Option 1: Upload trực tiếp
DATA_PATH = "/kaggle/input/result.csv"

# Option 2: Upload như dataset
# DATA_PATH = "/kaggle/input/your-dataset-name/result.csv"

# Option 3: Liệt kê files để tìm path đúng
# !ls /kaggle/input/
# ==========================================

# Load data
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Loaded from: {DATA_PATH}")
else:
    print(f"❌ File not found: {DATA_PATH}")
    print("\n📂 Available files:")
    !ls -la /kaggle/input/
    raise FileNotFoundError("Please update DATA_PATH")

# Extract documents
if 'processed_content' in df.columns:
    documents = df['processed_content'].dropna().astype(str).tolist()
elif 'content' in df.columns:
    documents = df['content'].dropna().astype(str).tolist()
else:
    print("Columns:", df.columns.tolist())
    raise ValueError("No content column found!")

# Filter empty documents
documents = [doc for doc in documents if len(doc.strip()) > 10]

print(f"\n📈 DATA STATS:")
print(f"   Total documents: {len(documents):,}")
print(f"   Avg length: {sum(len(d) for d in documents)/len(documents):.0f} chars")

# Sample preview
print("\n📝 SAMPLE DOCUMENTS:")
for i, doc in enumerate(documents[:3]):
    print(f"   {i+1}. {doc[:100]}...")

---
## 🧪 Cell 3: Compute PhoBERT Embeddings (CACHE)

**Mục đích:** Encode documents một lần duy nhất, reuse cho tất cả experiments

**⏱️ Time: 10-15 phút cho 10K docs**

**🔑 KEY INSIGHT:** 
- BERTopic có thể nhận pre-computed embeddings
- Compute embeddings 1 lần → Tiết kiệm 90% thời gian khi tuning

In [ ]:
# ===============================================
# CELL 3: COMPUTE PHOBERT EMBEDDINGS (CACHE)
# ===============================================
# FIX: Dùng SentenceTransformer (nhất quán với production bertopic_model.py)
#      thay vì AutoModel + manual mean pooling

import numpy as np
import time
import torch
from sentence_transformers import SentenceTransformer

print('🧪 COMPUTING PHOBERT EMBEDDINGS')
print('='*50)
print('⚠️  This will take 10-15 minutes but SAVES TIME later!')
print('   (Embeddings will be reused for ALL experiments)')

# ------------------------------------------------------------------
# [1/3] Load PhoBERT qua SentenceTransformer — NHẤT QUÁN với
#       VietnameseBERTopicModel.__init__() trong bertopic_model.py.
#       Không dùng AutoModel/AutoTokenizer vì mean pooling thủ công
#       cho vector space khác → kết quả tuning không phản ánh production.
# ------------------------------------------------------------------
print('\n[1/3] Loading PhoBERT via SentenceTransformer...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('vinai/phobert-base', device=device)
print(f'   ✅ PhoBERT loaded on {device}')

# ------------------------------------------------------------------
# [2/3] Sample documents — random để đảm bảo distribution
# ------------------------------------------------------------------
import random
random.seed(42)

TUNING_SAMPLE_SIZE = min(5000, len(documents))
tuning_docs = random.sample(documents, TUNING_SAMPLE_SIZE)  # FIX: random thay vì slice đầu
print(f'\n[2/3] Sample: {len(tuning_docs):,} docs (random seed=42)')

# ------------------------------------------------------------------
# [3/3] Encode — dùng .encode() của SentenceTransformer
#        normalize_embeddings=False vì BERTopic dùng UMAP cosine metric
# ------------------------------------------------------------------
print('\n[3/3] Computing embeddings...')
start_time = time.time()

embeddings = embedding_model.encode(
    tuning_docs,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
)

embed_time = (time.time() - start_time) / 60

# Giải phóng VRAM — model sẽ không cần encode nữa trong tuning loop
del embedding_model
torch.cuda.empty_cache()

print(f'\n✅ EMBEDDINGS CACHED')
print(f'   Shape : {embeddings.shape}')   # expected: (5000, 768)
print(f'   dtype : {embeddings.dtype}')   # expected: float32
print(f'   Time  : {embed_time:.1f} minutes')
print('='*50)
print('Ready for tuning experiments!')


---
## 🔬 Cell 4: Define Experiment Grid

**Mục đích:** Định nghĩa các hyperparameter combinations để thử nghiệm

**Lý thuyết:**
- **n_neighbors** (UMAP): Nhỏ → nhiều clusters, Lớn → ít clusters
- **min_dist** (UMAP): 0.0 → tight clusters, 0.1 → spread clusters
- **min_cluster_size** (HDBSCAN): Nhỏ → nhiều topics, Lớn → ít topics
- **min_samples** (HDBSCAN): Số core points tối thiểu
- **nr_topics**: 'auto' hoặc số cố định

In [ ]:
# ===============================================
# CELL 4: DEFINE EXPERIMENT GRID
# ===============================================

print("🔬 EXPERIMENT GRID")
print("="*50)

# Define experiments
experiments = [
    # Baseline (from Task 3.1)
    {
        'name': 'baseline',
        'n_neighbors': 15,
        'min_dist': 0.0,
        'min_cluster_size': 15,
        'min_samples': 10,
        'nr_topics': 'auto',
        'description': 'Task 3.1 default settings'
    },
    # Experiment 1: More topics (aggressive)
    {
        'name': 'exp_1_more_topics',
        'n_neighbors': 10,
        'min_dist': 0.0,
        'min_cluster_size': 10,
        'min_samples': 5,
        'nr_topics': 'auto',
        'description': 'More topics, local structure'
    },
    # Experiment 2: Conservative clustering
    {
        'name': 'exp_2_conservative',
        'n_neighbors': 10,
        'min_dist': 0.0,
        'min_cluster_size': 10,
        'min_samples': 10,
        'nr_topics': 'auto',
        'description': 'Conservative with more samples'
    },
    # Experiment 3: Global structure
    {
        'name': 'exp_3_global',
        'n_neighbors': 30,
        'min_dist': 0.0,
        'min_cluster_size': 15,
        'min_samples': 5,
        'nr_topics': 'auto',
        'description': 'Global structure, fewer outliers'
    },
    # Experiment 4: Fewer outliers
    {
        'name': 'exp_4_fewer_outliers',
        'n_neighbors': 30,
        'min_dist': 0.0,
        'min_cluster_size': 25,
        'min_samples': 10,
        'nr_topics': 'auto',
        'description': 'Fewer outliers, larger clusters'
    },
    # Experiment 5: Spread clusters
    {
        'name': 'exp_5_spread',
        'n_neighbors': 15,
        'min_dist': 0.1,
        'min_cluster_size': 15,
        'min_samples': 5,
        'nr_topics': 'auto',
        'description': 'Spread clusters (min_dist=0.1)'
    },
    # Experiment 6: Spread + more topics
    {
        'name': 'exp_6_spread_many',
        'n_neighbors': 15,
        'min_dist': 0.1,
        'min_cluster_size': 10,
        'min_samples': 5,
        'nr_topics': 'auto',
        'description': 'Spread + smaller clusters'
    },
    # Experiment 7: Force 10 topics
    {
        'name': 'exp_7_force_10',
        'n_neighbors': 15,
        'min_dist': 0.0,
        'min_cluster_size': 15,
        'min_samples': 5,
        'nr_topics': 10,
        'description': 'Force exactly 10 topics'
    },
    # Experiment 8: Force 15 topics
    {
        'name': 'exp_8_force_15',
        'n_neighbors': 15,
        'min_dist': 0.0,
        'min_cluster_size': 15,
        'min_samples': 5,
        'nr_topics': 15,
        'description': 'Force exactly 15 topics'
    },
    # Experiment 9: Most conservative
    {
        'name': 'exp_9_most_conservative',
        'n_neighbors': 30,
        'min_dist': 0.1,
        'min_cluster_size': 25,
        'min_samples': 10,
        'nr_topics': 'auto',
        'description': 'Most conservative settings'
    },
    # Experiment 10: Most aggressive
    {
        'name': 'exp_10_most_aggressive',
        'n_neighbors': 10,
        'min_dist': 0.0,
        'min_cluster_size': 5,
        'min_samples': 3,
        'nr_topics': 'auto',
        'description': 'Most aggressive (many small topics)'
    },
]

# Print summary
print(f"Total experiments: {len(experiments)}")
print("\n" + "-"*80)
print(f"{'Name':<25} {'n_nbr':>6} {'m_dist':>6} {'m_clust':>7} {'m_samp':>6} {'nr_top':>6}")
print("-"*80)
for exp in experiments:
    print(f"{exp['name']:<25} {exp['n_neighbors']:>6} {exp['min_dist']:>6} {exp['min_cluster_size']:>7} {exp['min_samples']:>6} {str(exp['nr_topics']):>6}")
print("-"*80)

---
## 🏋️ Cell 5: Run Experiments

**Mục đích:** Chạy tất cả experiments và thu thập metrics

**⏱️ Time: 30-60 phút cho 10 experiments**

**Metrics thu thập:**
- n_topics_found: Số topics tìm được
- n_outliers: Số documents là outliers
- outlier_ratio: Tỷ lệ outliers
- coherence_cv: Coherence score (nếu có)
- diversity: Topic diversity score
- training_time: Thời gian train (giây)

In [ ]:
# ===============================================
# CELL 5: RUN EXPERIMENTS
# ===============================================

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import time
import warnings
warnings.filterwarnings('ignore')

print('🏋️ RUNNING EXPERIMENTS')
print('='*50)
print(f'Documents: {len(tuning_docs):,}')
print(f'Embeddings shape: {embeddings.shape}')
print(f'Experiments: {len(experiments)}')
print('='*50)

results = []


def calculate_diversity(topic_model, top_n=10):
    """Calculate topic diversity (unique words ratio)"""
    try:
        topics = topic_model.get_topics()
        all_words = []
        for topic_id, words in topics.items():
            if topic_id != -1:
                all_words.extend([w[0] for w in words[:top_n]])
        if not all_words:
            return 0.0
        return len(set(all_words)) / len(all_words)
    except Exception:
        return 0.0


def calculate_umass(topic_model, texts):
    """
    Tính coherence U_Mass — nhanh hơn C_V (không cần sliding window).
    Dùng trong tuning loop; C_V chỉ tính cho best config cuối cùng.
    U_Mass range: thường âm, gần 0 hơn = tốt hơn.
    """
    try:
        topics_dict = {tid: words for tid, words in topic_model.get_topics().items() if tid != -1}
        if not topics_dict:
            return 0.0
        topics_words = [
            [w for w, _ in words[:10]]
            for words in topics_dict.values()
        ]
        # Tokenize — filter token độ dài <= 1 (nhất quán với bertopic_model.py)
        tokenized = [[t for t in doc.split() if len(t) > 1] for doc in texts]
        tokenized = [t for t in tokenized if t]
        dictionary = Dictionary(tokenized)
        dictionary.filter_extremes(no_below=2, no_above=0.95)
        cm = CoherenceModel(
            topics=topics_words,
            texts=tokenized,
            dictionary=dictionary,
            coherence='u_mass',
        )
        return cm.get_coherence()
    except Exception:
        return 0.0


for i, exp in enumerate(experiments):
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(experiments)}] {exp['name']}")
    print(f"   {exp['description']}")
    print(f"   n_neighbors={exp['n_neighbors']}, min_dist={exp['min_dist']}, "
          f"min_cluster_size={exp['min_cluster_size']}, min_samples={exp['min_samples']}, "
          f"nr_topics={exp['nr_topics']}")

    start_time = time.time()

    try:
        umap_model = UMAP(
            n_neighbors=exp['n_neighbors'],
            n_components=5,
            min_dist=exp['min_dist'],
            metric='cosine',
            random_state=42,
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=exp['min_cluster_size'],
            min_samples=exp['min_samples'],
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
        )

        # FIX: cast nr_topics sang int nếu là số, tránh TypeError trên một số BERTopic versions
        nr_topics_val = None if exp['nr_topics'] == 'auto' else int(exp['nr_topics'])

        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            nr_topics=nr_topics_val,
            calculate_probabilities=False,  # Faster trong tuning
            verbose=False,
        )

        # Dùng pre-computed embeddings — bỏ qua encode PhoBERT
        topics, probs = topic_model.fit_transform(tuning_docs, embeddings=embeddings)

        training_time = time.time() - start_time

        topic_info = topic_model.get_topic_info()
        n_topics = len([t for t in topic_info['Topic'] if t != -1])
        n_outliers = (
            topic_info[topic_info['Topic'] == -1]['Count'].sum()
            if -1 in topic_info['Topic'].values else 0
        )
        outlier_ratio = n_outliers / len(tuning_docs)
        diversity = calculate_diversity(topic_model)
        coherence_umass = calculate_umass(topic_model, tuning_docs)  # FIX: thêm metric

        result = {
            'name': exp['name'],
            'description': exp['description'],
            'n_neighbors': exp['n_neighbors'],
            'min_dist': exp['min_dist'],
            'min_cluster_size': exp['min_cluster_size'],
            'min_samples': exp['min_samples'],
            'nr_topics': str(exp['nr_topics']),
            'n_topics_found': n_topics,
            'n_outliers': n_outliers,
            'outlier_ratio': outlier_ratio,
            'diversity': diversity,
            'coherence_umass': coherence_umass,  # FIX: thêm vào results
            'training_time': training_time,
        }
        results.append(result)

        print(f'   ✅ {training_time:.1f}s | Topics: {n_topics} | '
              f'Outliers: {outlier_ratio*100:.1f}% | '
              f'Diversity: {diversity:.3f} | '
              f'U_Mass: {coherence_umass:.4f}')

        pd.DataFrame(results).to_csv('/kaggle/working/tuning_progress.csv', index=False)

    except Exception as e:
        print(f'   ❌ FAILED: {e}')
        results.append({
            'name': exp['name'],
            'description': exp['description'],
            'error': str(e),
        })

print('\n' + '='*50)
print(f'✅ COMPLETED {len(results)} EXPERIMENTS')
print('='*50)


---
## 📊 Cell 6: Analyze Results

**Mục đích:** So sánh experiments và chọn best config

**Tiêu chí chọn:**
1. Outlier ratio < 20%
2. Diversity > 0.5
3. N topics trong range hợp lý (5-15)

In [ ]:
# ===============================================
# CELL 6: ANALYZE RESULTS
# ===============================================

import pandas as pd

print('📊 EXPERIMENT RESULTS ANALYSIS')
print('='*60)

results_df = pd.DataFrame(results)
valid_results = results_df[results_df['n_topics_found'].notna()].copy()


def composite_score(row):
    """
    Composite score để rank experiments.

    Weights:
        outlier_ratio   → 35% (fewer outliers = better)
        diversity       → 25% (more diverse = better)
        coherence_umass → 25% (U_Mass gần 0 = better; normalize từ âm về [0,1])
        topic_count     → 15% (smooth penalty, peak tại 10 topics)

    FIX so với phiên bản cũ:
        - Thêm coherence_umass thay vì static topic_bonus
        - Smooth penalty topic_count thay vì hard threshold
        - Normalize U_Mass: clip [-20, 0] → [0, 1]
    """
    outlier_score = (1 - row['outlier_ratio']) * 0.35
    diversity_score = row['diversity'] * 0.25

    # U_Mass thường âm, gần 0 hơn = tốt hơn
    # Normalize về [0, 1]: clip [-20, 0] → map tuyến tính
    u_mass_clipped = max(-20.0, min(0.0, row.get('coherence_umass', -20.0)))
    coherence_score = (u_mass_clipped + 20.0) / 20.0 * 0.25

    # Smooth penalty: peak = 1.0 tại 10 topics, giảm dần ra xa
    n = row['n_topics_found']
    topic_score = max(0.0, 1.0 - abs(n - 10) / 20.0) * 0.15

    return outlier_score + diversity_score + coherence_score + topic_score


valid_results['composite_score'] = valid_results.apply(composite_score, axis=1)
valid_results = valid_results.sort_values('composite_score', ascending=False)

print('\n📋 ALL RESULTS (sorted by composite score):')
print('-'*110)
display_cols = ['name', 'n_topics_found', 'outlier_ratio', 'diversity', 'coherence_umass', 'composite_score', 'training_time']
print(valid_results[display_cols].to_string(index=False))
print('-'*110)

print('\n🏆 TOP 3 CONFIGURATIONS:')
for i, (_, row) in enumerate(valid_results.head(3).iterrows()):
    print(f"\n#{i+1}: {row['name']}")
    print(f"    {row['description']}")
    print(f"    Topics: {row['n_topics_found']} | "
          f"Outliers: {row['outlier_ratio']*100:.1f}% | "
          f"Diversity: {row['diversity']:.3f} | "
          f"U_Mass: {row['coherence_umass']:.4f}")
    print(f"    n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, "
          f"min_cluster_size={row['min_cluster_size']}, min_samples={row['min_samples']}")
    print(f"    Composite Score: {row['composite_score']:.4f}")

best_config = valid_results.iloc[0]
print('\n' + '='*60)
print(f"🥇 BEST CONFIG: {best_config['name']}")
print('='*60)


---
## 🏆 Cell 7: Train Final Model with Best Config

**Mục đích:** Train BERTopic với best hyperparameters trên full dataset

**⏱️ Time: 5-15 phút tùy dataset size**

In [ ]:
# ===============================================
# CELL 7: TRAIN FINAL MODEL WITH BEST CONFIG
# ===============================================

print("🏆 TRAINING FINAL MODEL")
print("="*50)
print(f"Using config: {best_config['name']}")
print(f"Documents: {len(tuning_docs):,}")

# Extract best hyperparameters
best_params = {
    'n_neighbors': int(best_config['n_neighbors']),
    'min_dist': float(best_config['min_dist']),
    'min_cluster_size': int(best_config['min_cluster_size']),
    'min_samples': int(best_config['min_samples']),
    'nr_topics': best_config['nr_topics']
}

print(f"\nHyperparameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# Create final model
print("\n[1/4] Creating UMAP model...")
umap_model = UMAP(
    n_neighbors=best_params['n_neighbors'],
    n_components=5,
    min_dist=best_params['min_dist'],
    metric='cosine',
    random_state=42
)

print("[2/4] Creating HDBSCAN model...")
hdbscan_model = HDBSCAN(
    min_cluster_size=best_params['min_cluster_size'],
    min_samples=best_params['min_samples'],
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

print("[3/4] Creating BERTopic model...")
nr_topics = None if best_params['nr_topics'] == 'auto' else int(best_params['nr_topics'])
final_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=nr_topics,
    calculate_probabilities=True,
    verbose=True
)

print("[4/4] Fitting model...")
start_time = time.time()
topics, probs = final_model.fit_transform(tuning_docs, embeddings=embeddings)
training_time = time.time() - start_time

# Results
topic_info = final_model.get_topic_info()
n_topics = len([t for t in topic_info['Topic'] if t != -1])
n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].sum() if -1 in topic_info['Topic'].values else 0

print("\n" + "="*50)
print("✅ FINAL MODEL TRAINED!")
print("="*50)
print(f"Topics found: {n_topics}")
print(f"Outliers: {n_outliers} ({n_outliers/len(tuning_docs)*100:.1f}%)")
print(f"Training time: {training_time:.1f} seconds")

# Show topics
print("\n📋 TOPIC SUMMARY:")
print(topic_info[['Topic', 'Count', 'Name']].head(15).to_string(index=False))

---
## 💾 Cell 8: Save Results

**Mục đích:** Lưu model và kết quả tuning

**Output files:**
- `tuning_results.csv` - All experiments results
- `best_config.json` - Best hyperparameters
- `bertopic_tuned_model/` - Model files
- `tuned_topics.csv` - Topics from best model

In [ ]:
# ===============================================
# CELL 8: SAVE RESULTS
# ===============================================

import json
import pickle
import os

print("💾 SAVING RESULTS")
print("="*50)

# Create output directory
OUTPUT_DIR = "/kaggle/working/output_task_3.2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save tuning results
results_path = f"{OUTPUT_DIR}/tuning_results.csv"
valid_results.to_csv(results_path, index=False)
print(f"✅ Saved: {results_path}")

# 2. Save best config
best_config_dict = {
    'name': best_config['name'],
    'description': best_config['description'],
    'hyperparameters': best_params,
    'metrics': {
        'n_topics_found': int(best_config['n_topics_found']),
        'outlier_ratio': float(best_config['outlier_ratio']),
        'diversity': float(best_config['diversity']),
        'composite_score': float(best_config['composite_score'])
    }
}
config_path = f"{OUTPUT_DIR}/best_config.json"
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(best_config_dict, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {config_path}")

# 3. Save model
model_path = f"{OUTPUT_DIR}/bertopic_tuned_model"
final_model.save(model_path, serialization="pickle")
print(f"✅ Saved: {model_path}/")

# 4. Save topics
topics_path = f"{OUTPUT_DIR}/tuned_topics.csv"
topic_info.to_csv(topics_path, index=False)
print(f"✅ Saved: {topics_path}")

# 5. Save summary
summary = {
    'task': '3.2 - BERTopic Tuning',
    'n_documents': len(tuning_docs),
    'n_experiments': len(experiments),
    'best_config': best_config_dict,
    'final_metrics': {
        'n_topics': n_topics,
        'n_outliers': n_outliers,
        'outlier_ratio': n_outliers / len(tuning_docs),
        'training_time_seconds': training_time
    }
}
summary_path = f"{OUTPUT_DIR}/tuning_summary.pkl"
with open(summary_path, 'wb') as f:
    pickle.dump(summary, f)
print(f"✅ Saved: {summary_path}")

# List output files
print("\n" + "="*50)
print("📂 OUTPUT FILES:")
print("="*50)
!ls -la {OUTPUT_DIR}/

print("\n" + "="*50)
print("🎉 TASK 3.2 COMPLETED!")
print("="*50)
print(f"\n📥 Download from Output tab: {OUTPUT_DIR}/")

---
## 📈 (Optional) Cell 9: Visualizations

**Mục đích:** Tạo charts để compare experiments

In [ ]:
# ===============================================
# CELL 9: VISUALIZATIONS (OPTIONAL)
# ===============================================

import matplotlib.pyplot as plt

print("📈 GENERATING VISUALIZATIONS")
print("="*50)

# 1. Bar chart: Composite scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Composite Score
ax1 = axes[0, 0]
colors = ['green' if x == valid_results['composite_score'].max() else 'steelblue' 
          for x in valid_results['composite_score']]
ax1.barh(valid_results['name'], valid_results['composite_score'], color=colors)
ax1.set_xlabel('Composite Score')
ax1.set_title('Experiment Ranking (Higher = Better)')
ax1.invert_yaxis()

# Plot 2: Outlier Ratio
ax2 = axes[0, 1]
colors = ['red' if x > 0.2 else 'green' if x < 0.15 else 'orange' 
          for x in valid_results['outlier_ratio']]
ax2.barh(valid_results['name'], valid_results['outlier_ratio']*100, color=colors)
ax2.axvline(x=20, color='red', linestyle='--', label='Threshold (20%)')
ax2.set_xlabel('Outlier Ratio (%)')
ax2.set_title('Outlier Ratio (Lower = Better)')
ax2.legend()
ax2.invert_yaxis()

# Plot 3: Topic Diversity
ax3 = axes[1, 0]
colors = ['green' if x > 0.7 else 'orange' if x > 0.5 else 'red' 
          for x in valid_results['diversity']]
ax3.barh(valid_results['name'], valid_results['diversity'], color=colors)
ax3.axvline(x=0.5, color='orange', linestyle='--', label='Min Acceptable (0.5)')
ax3.set_xlabel('Diversity Score')
ax3.set_title('Topic Diversity (Higher = Better)')
ax3.legend()
ax3.invert_yaxis()

# Plot 4: Number of Topics
ax4 = axes[1, 1]
colors = ['green' if 5 <= x <= 15 else 'orange' 
          for x in valid_results['n_topics_found']]
ax4.barh(valid_results['name'], valid_results['n_topics_found'], color=colors)
ax4.axvline(x=5, color='green', linestyle='--', alpha=0.5)
ax4.axvline(x=15, color='green', linestyle='--', alpha=0.5)
ax4.set_xlabel('Number of Topics')
ax4.set_title('Topics Found (5-15 = Ideal)')
ax4.invert_yaxis()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/tuning_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {OUTPUT_DIR}/tuning_comparison.png")

---
## ✅ Summary

### What we did:
1. Loaded preprocessed data from Task 2
2. Pre-computed PhoBERT embeddings (reused across all experiments)
3. Ran 10 hyperparameter experiments
4. Selected best config based on composite score
5. Trained final model with best hyperparameters
6. Saved all results for Task 3.3 comparison

### Output files:
- `tuning_results.csv` - All experiments comparison
- `best_config.json` - Best hyperparameters to use
- `bertopic_tuned_model/` - Tuned model files
- `tuned_topics.csv` - Topics from tuned model
- `tuning_comparison.png` - Visualization

### Next steps (Task 3.3):
- Load LDA results from Task 2.2
- Compare LDA vs BERTopic coherence scores
- Qualitative analysis of topics
- Write comparison report